# Training a Potential: how to setup data pipeline, model and trainer


# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp


## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data.


In [1]:
%load_ext autoreload
%autoreload 2
import logging, sys
logger = logging.getLogger("molpot")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))

import molpot as mpot
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path
from molpot import alias

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    total=12,
    processes=[mpot.process.NeighborList(cutoff=5.0)],
)

Parsing molecule aspirin


Loaded 12 frames


In [3]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 12

                     dataset: rMD17                     
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [4]:
train_ds, eval_ds = torch.utils.data.random_split(rmd17_ds, [8, 4])
train_dl = mpot.DataLoader(train_ds, num_workers=0, drop_last=True)
eval_dl = mpot.DataLoader(eval_ds)

## Setting up the model


In [5]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1,
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy"),
    n_neurons=[64, 1],
    reduce="sum",
)
f_readout = mpot.potential.nnp.readout.Derivative(
    fx_key=("predicts", "energy"), dx_key=alias.pair_diff, to_key=("predicts", "forces"), 
)
potential = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)
# potential.append_post_process(f_readout)

In [11]:
for i, batch in enumerate(train_dl):
    print(f'{i} batch')
    batch = torch.vmap(pinet)(batch)
    batch = torch.vmap(e_readout)(batch)
    batch = f_readout(batch)

0 batch
1 batch
2 batch
3 batch
4 batch
5 batch
6 batch
7 batch


In [33]:
dfdx = torch.autograd.grad(
    batch['predicts', 'energy'],
    batch['pairs', 'dist'],
    create_graph=False,
    retain_graph=False,
)
dfdx

(tensor([[ 8.6299e-01, -1.3932e+00,  1.2773e+00,  1.3063e+00,  9.0097e-01,
          -1.3922e+00, -6.6922e-01, -1.4872e+00,  1.4078e+00, -7.3837e-01,
          -3.6102e-01, -1.2627e-01,  1.3160e+00,  1.1702e+00, -1.2324e+00,
           1.1518e+00,  7.8364e-02, -2.9168e+00,  3.0507e-02, -1.5601e+00,
          -7.9590e-02,  4.3970e-03, -6.0345e-01,  1.9148e+00,  1.4972e+00,
           4.5137e-01,  1.7830e-01,  2.1971e+00, -1.1210e+00,  1.2922e+00,
           3.0622e-01,  8.1302e-01, -1.7674e-01, -1.8401e-01,  1.6841e+00,
          -4.2799e-01,  2.6466e+00,  1.0629e+00, -1.0148e+00,  4.9703e-01,
           1.0776e-01, -2.5688e-01, -1.7875e+00,  2.7555e+00,  2.1975e+00,
           1.3190e-01, -5.4643e+00,  7.0635e-02,  4.6516e-01,  1.9421e-01,
          -4.4682e+00, -2.0440e-01,  2.6158e+00, -1.8700e-01,  1.5595e+00,
           1.1946e-02,  1.4345e-01, -3.9937e-01,  9.4815e-01, -1.5937e-01,
          -1.8890e-01,  2.2134e+00,  1.8851e+00, -4.5930e+00, -9.7664e-01,
           2.5899e+00,  1

In [31]:
import torch
from torch import nn
from tensordict import TensorDict


class AComplexModel(nn.Module):

    def __call__(self, td):
        print("call")
        energy = torch.square(td["dist"])  # per dist
        td["energy"] = energy
        return td

n_batch = 4
td = TensorDict(
    {"dist": torch.randint(0, 10, (n_batch, 10), dtype=torch.float32, requires_grad=True),
     "inrelative": torch.randint(0, 10, (n_batch, 10), dtype=torch.float32),},
    batch_size=n_batch,  # other tensor not involved in grad
)
model = AComplexModel()
vmap_model = torch.vmap(model)
td = vmap_model(td)
print(td["dist"])
print(td["energy"])
print(td["inrelative"])

# Q1: Can I get index of key of tensordict in this way,
#     and use it in `argnums`?
dist_index = list(td.keys()).index("dist")
print(dist_index)

# Q2: I only want to get `energy` tensor's grad
#     but `inrelative` is zeros-like tensor, is it ok?
# Q3: If I want to use model in traditional `backward` way in the training,
#     and also want to calculate a certain tensor's grad in this way,
#     is that means the forward pass should be done twice?
grad = torch.func.grad(lambda td: model(td)["energy"].sum(), argnums=dist_index)
grad_td = grad(td[0])
print(grad_td["dist"])
print(grad_td["energy"])
print(grad_td["inrelative"])

vmap_grad = torch.vmap(grad)
batched_grad_td = vmap_grad(td)
print(batched_grad_td["dist"])
print(batched_grad_td["energy"])
print(batched_grad_td["inrelative"])

call
tensor([[1., 0., 1., 6., 0., 2., 7., 2., 0., 6.],
        [9., 8., 4., 3., 2., 0., 3., 6., 2., 8.],
        [8., 3., 7., 9., 1., 6., 4., 5., 4., 9.],
        [1., 0., 4., 1., 0., 2., 1., 1., 4., 2.]], requires_grad=True)
tensor([[ 1.,  0.,  1., 36.,  0.,  4., 49.,  4.,  0., 36.],
        [81., 64., 16.,  9.,  4.,  0.,  9., 36.,  4., 64.],
        [64.,  9., 49., 81.,  1., 36., 16., 25., 16., 81.],
        [ 1.,  0., 16.,  1.,  0.,  4.,  1.,  1., 16.,  4.]],
       grad_fn=<PowBackward0>)
tensor([[8., 4., 2., 3., 3., 5., 9., 2., 6., 1.],
        [2., 1., 1., 8., 9., 4., 5., 8., 6., 2.],
        [0., 0., 2., 2., 1., 1., 5., 9., 0., 3.],
        [8., 0., 5., 3., 4., 6., 9., 5., 5., 7.]])
0
call
tensor([ 2.,  0.,  2., 12.,  0.,  4., 14.,  4.,  0., 12.],
       grad_fn=<MulBackward0>)
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
call
tensor([[ 2.,  0.,  2., 12.,  0.,  4., 14.,  4.,  0., 12.],
        [18., 16.,  8.,  6.,  4.,  0.,  6

In [33]:
import torch
from torch.func import grad, vmap

# 假设有一个标量函数 f(params, x)
def f(params, x):
    # 可以是任意可微操作
    print('call f')
    return (params * x).sum()

# 定义对 params 求梯度的函数
grad_f = grad(f, argnums=0)  # 只对第0个参数(即 params)求导

# 将 grad_f 对 x 做向量化 vmap
batched_grad_f = vmap(grad_f, in_dims=(None, 0))

# 准备数据
params = torch.tensor([2.0, 3.0], requires_grad=True)  # shape [2]
x = torch.randn(5, 2)  # batch=5，每个样本是 shape [2]

# 计算：对每个 x[i]，求 f(params, x[i]) 对 params 的梯度
result = batched_grad_f(params, x)
print(result)  # shape [5, 2]

call f
tensor([[ 0.2150, -0.2461],
        [-0.2986, -0.3519],
        [-2.0676,  1.1172],
        [ 0.3881, -0.6173],
        [-1.0419, -0.0282]])


In [13]:
from tensordict import TensorDict
from tensordict.nn import TensorDictModule as Mod
import torch

mod = Mod(lambda x: (x+1).mean(), in_keys=["a"], out_keys=["b"])

td = TensorDict(a=torch.randn(10, 11, requires_grad=True), batch_size=[10])

vmap_mod = torch.vmap(mod, (0,))
td_out = vmap_mod(td)
print(td_out)

grad = torch.func.grad(lambda td: mod(td)["b"])
print(grad(td[0])["a"])

vmap_grad = torch.vmap(grad, (0,))
td_out = vmap_grad(td)
print(td_out["a"])

TensorDict(
    fields={
        a: Tensor(shape=torch.Size([10, 11]), device=cpu, dtype=torch.float32, is_shared=False),
        b: Tensor(shape=torch.Size([10]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([10]),
    device=None,
    is_shared=False)
tensor([0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
        0.0909, 0.0909])
tensor([[0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
         0.0909, 0.0909],
        [0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
         0.0909, 0.0909],
        [0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
         0.0909, 0.0909],
        [0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
         0.0909, 0.0909],
        [0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909,
         0.0909, 0.0909],
        [0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 0.0909, 

In [9]:
save_dir = Path("pinet2-rmd17")

optimizer = torch.optim.Adam(potential.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
loss_fn = mpot.engine.MultiTargetLoss(
    torch.nn.MSELoss(), [("energy", "energy", 1.0), ("forces", "forces", 10.0)]
)

potential_trainer = mpot.PotentialTrainer(
    model=potential,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device="cpu",
    no_grad_eval=False,
)
potential_trainer.add_lr_scheduler(scheduler)
potential_trainer.add_checkpoint(save_dir)
potential_trainer.attach_progressbar()
potential_trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(lambda pred, label: (pred["energy"], label["energy"])),
    BatchWise(),
    target="all",
)
potential_trainer.add_metric(
    "f_mae",
    MeanAbsoluteError(lambda pred, label: (pred["forces"], label["forces"])),
    BatchWise(),
    target="all",
)
potential_trainer.attach_tensorboard(
    save_dir / "tb",
)
state = potential_trainer.run(train_data=train_dl, max_steps=1000, eval_data=eval_dl)

Current run is terminating due to exception: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.
Engine run is terminating due to exception: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.


call potential


ValueError: vmap(modules, in_dims=0, ...)(<inputs>):
Got in_dim=0 for some input, but that input is a Tensor
of dimensionality 0 so expected in_dim to satisfy
-0 <= in_dim < 0.